In [1]:
import pandas as pd
import numpy as np

In [4]:
# Read csv files
pnrs = pd.read_csv('PRMI_DM_ALL_PNRs.csv')
targets = pd.read_csv('PRMI-DM_TARGET_FLIGHTS.csv')
flights = pd.read_csv('PRMI-DM-AVAILABLE_FLIGHTS.csv')

# Quick edits to data
# Aggregate seat counts
targets['AVAIL_CNT'] = targets['C_AVAIL_CNT'] + targets['Y_AVAIL_CNT']
flights['AVAIL_CNT'] = flights['C_AVAIL_CNT'] + flights['Y_AVAIL_CNT']
# Drop unnecessary columns
pnrs = pnrs.drop(columns=['CREATION_DTZ', 'CABIN_CD', 'COS_CD', 'OPER_OD_ORIG_CD', 'DEP_DT', 'FLT_NUM', 'DEP_DTML', 'ARR_DTML'])
targets = targets.drop(columns=['CHANGE_TYPE', 'DEP_DT', 'DEP_DTML', 'ARR_DTML', 'C_CAP_CNT', 'C_AUL_CNT', 'C_PAX_CNT', 'Y_CAP_CNT', 'Y_AUL_CNT', 'Y_PAX_CNT', 'C_AVAIL_CNT', 'Y_AVAIL_CNT'])
flights = flights.drop(columns=['DEP_DT', 'DEP_DTML', 'ARR_DTML', 'C_CAP_CNT', 'C_AUL_CNT', 'C_PAX_CNT', 'Y_CAP_CNT', 'Y_AUL_CNT', 'Y_PAX_CNT', 'C_AVAIL_CNT', 'Y_AVAIL_CNT'])
# Convert DTMZ columns to datetime
pnrs['DEP_DTMZ'] = pd.to_datetime(pnrs['DEP_DTMZ'], format='%Y-%m-%d %H:%M:%S')
pnrs['ARR_DTMZ'] = pd.to_datetime(pnrs['ARR_DTMZ'], format='%Y-%m-%d %H:%M:%S')
# targets['DEP_DTMZ'] = pd.to_datetime(targets['DEP_DTMZ'], format='%Y-%m-%d %H:%M:%S')
# targets['ARR_DTMZ'] = pd.to_datetime(targets['ARR_DTMZ'], format='%Y-%m-%d %H:%M:%S')
targets['DEP_DTMZ'] = pd.to_datetime(targets['DEP_DTMZ'], errors='raise')
targets['ARR_DTMZ'] = pd.to_datetime(targets['ARR_DTMZ'], errors='raise')

flights['DEP_DTMZ'] = pd.to_datetime(flights['DEP_DTMZ'], format='%Y-%m-%d %H:%M:%S')
flights['ARR_DTMZ'] = pd.to_datetime(flights['ARR_DTMZ'], format='%Y-%m-%d %H:%M:%S')

# Check validity
pnrs.info()
targets.info()
flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43832 entries, 0 to 43831
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   RECLOC           43832 non-null  int64         
 1   OPER_OD_DEST_CD  43832 non-null  object        
 2   DEP_KEY          43832 non-null  object        
 3   ORIG_CD          43832 non-null  object        
 4   DEST_CD          43832 non-null  object        
 5   DEP_DTMZ         43832 non-null  datetime64[ns]
 6   ARR_DTMZ         43832 non-null  datetime64[ns]
 7   OD_BROKEN_IND    43832 non-null  int64         
 8   PAX_CNT          43832 non-null  int64         
 9   CVM              43832 non-null  float64       
 10  CONN_TIME_MINS   18059 non-null  float64       
dtypes: datetime64[ns](2), float64(2), int64(3), object(4)
memory usage: 3.7+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 7 columns):
 #   Column     N

In [5]:
def create_route(orig, dest, start_time, path):
    '''
    Generates possible routes by recursively searching flights dataset until end destination is reached
    Fails if no route is found in under 48 hours from start_time
    Returns a list of all found routes
    '''
    routes = []
    test_path = path.copy() # Copy the path
    path_cities = []
    # Find earliest departure time for flights
    if len(test_path) > 0:
        last_flight_time = flights[flights['DEP_KEY'] == test_path[-1]].iloc[0]['ARR_DTMZ']
    else:
        last_flight_time = pd.Timestamp(year=2000, month=1, day=1)

    for flight in test_path:
        path_cities.append(flights[flights['DEP_KEY'] == flight].iloc[0]['ORIG_CD'])

    filtered_flights = flights[flights['ORIG_CD'] == orig] # Filter wrong origin codes
    filtered_flights = filtered_flights[~filtered_flights['DEST_CD'].isin(path_cities)] # Filter flights that go to a previously visited city
    # filtered_flights = filtered_flights[filtered_flights['DEP_DTMZ'] > last_flight_time] # Filter too early flights

    # Filter too early or late flights
    indices_to_drop = []
    for idx, flight in filtered_flights.iterrows():
        if last_flight_time.year == 2000:
            connection_hours = 1
        else:
            connection_hours = (flight['DEP_DTMZ'] - last_flight_time).value / 3.6e12 # Convert connection miliseconds to hours
        delay_hours = (flight['ARR_DTMZ'] - start_time).value / 3.6e12 # Convert total reroute miliseconds to hours
        if flight['DEP_DTMZ'] < start_time or delay_hours > 48 or connection_hours < 1:
            indices_to_drop.append(idx)
    filtered_flights = filtered_flights.drop(indices_to_drop)

    # Find new flights
    test_path.append('') # Add element to replace with test cities
    for idx, flight in filtered_flights.iterrows():
        test_path[-1] = flight['DEP_KEY'] # Insert flight into path being tested

        
        # if flight['DEST_CD'] == dest: # Destination found
        #     routes.append(test_path.copy())
        #     continue

        if flight['DEST_CD'] == dest:  # Destination found
            completed_route = test_path.copy()
        
            # First and last flight keys
            first_flight = completed_route[0]
            last_flight = completed_route[-1]
        
            # Get times
            dep_time_first = flights.loc[
                flights['DEP_KEY'] == first_flight, 'DEP_DTMZ'
            ].iloc[0]
        
            arr_time_last = flights.loc[
                flights['DEP_KEY'] == last_flight, 'ARR_DTMZ'
            ].iloc[0]
        
            # Append times to route
            completed_route.append(dep_time_first)
            completed_route.append(arr_time_last)
        
            routes.append(completed_route)
            continue

        
        # Recursive search
        subroutes = create_route(flight['DEST_CD'], dest, start_time, test_path)
        if len(subroutes) > 0:
            for row in subroutes:
                routes.append(row)

    return routes

In [6]:
alt_routes = []

# Remove cancelled flights from the dataset
for key in targets['DEP_KEY']:
    flights = flights[flights['DEP_KEY'] != key]

# Iterate through cancelled flights
# Find new routes leading to destination city
# Skip route as soon as it takes >48H
# Store routes in a list
for idx, irop in targets.iterrows():
    routes = create_route(irop['ORIG_CD'], irop['DEST_CD'], irop['DEP_DTMZ'], [])
    for row in routes:
        alt_routes.append(row)

In [7]:
# Remove duplicate routes
# print(alt_routes)
alt_routes = [list(row) for row in set(tuple(row) for row in alt_routes)]
print(alt_routes)

[['AZ20271022TPHQHP10566', Timestamp('2027-10-23 00:03:00'), Timestamp('2027-10-23 03:11:00')], ['AZ20271021TPHEID11017', Timestamp('2027-10-22 02:39:00'), Timestamp('2027-10-22 04:20:00')], ['AZ20271023RBRTPH14413', Timestamp('2027-10-23 15:55:00'), Timestamp('2027-10-23 17:35:00')], ['AZ20271023PTPTPH16212', Timestamp('2027-10-23 15:06:00'), Timestamp('2027-10-23 21:38:00')], ['AZ20271021TPHCEF5474', Timestamp('2027-10-22 02:48:00'), Timestamp('2027-10-22 04:50:00')], ['AZ20271022TZJTPH2996', Timestamp('2027-10-22 15:54:00'), Timestamp('2027-10-22 17:44:00')], ['AZ20271022TPHQXG14385', 'AZ20271023QXGVUY8894', Timestamp('2027-10-22 18:40:00'), Timestamp('2027-10-23 23:13:00')], ['AZ20271023NWYTPH13488', Timestamp('2027-10-24 03:56:00'), Timestamp('2027-10-24 07:37:00')], ['AZ20271020TPHERW5969', Timestamp('2027-10-21 05:37:00'), Timestamp('2027-10-21 08:06:00')], ['AZ20271020TPHQHP16158', Timestamp('2027-10-21 08:38:00'), Timestamp('2027-10-21 11:46:00')], ['AZ20271022TPHLIZ1248', Tim

In [26]:
import json

file_path = "alternate_flight_paths.json"

with open(file_path, "w") as file:
    json.dump(alt_routes, file, indent=4)

In [9]:
pax_list = (
    pnrs[['PAX_CNT', 'CVM', 'DEP_DTMZ', 'ARR_DTMZ']]
    .values
    .tolist()
)

print(pax_list[:10])

[[1, 3.32192809488736, Timestamp('2027-10-22 19:06:00'), Timestamp('2027-10-22 21:35:00')], [3, 5.68904982996268, Timestamp('2027-10-21 14:38:00'), Timestamp('2027-10-21 21:29:00')], [3, 5.68904982996268, Timestamp('2027-10-21 23:25:00'), Timestamp('2027-10-22 02:55:00')], [3, 5.68904982996268, Timestamp('2027-11-05 00:57:00'), Timestamp('2027-11-05 04:25:00')], [3, 5.68904982996268, Timestamp('2027-11-05 05:51:00'), Timestamp('2027-11-05 12:53:00')], [1, 5.85570990038199, Timestamp('2027-10-21 11:07:00'), Timestamp('2027-10-21 17:52:00')], [1, 5.85570990038199, Timestamp('2027-10-21 23:07:00'), Timestamp('2027-10-22 06:19:00')], [1, 5.85570990038199, Timestamp('2027-11-07 11:19:00'), Timestamp('2027-11-07 17:43:00')], [1, 5.85570990038199, Timestamp('2027-11-08 02:51:00'), Timestamp('2027-11-08 09:51:00')], [3, 5.18487546114268, Timestamp('2027-10-21 10:36:00'), Timestamp('2027-10-21 17:21:00')]]


In [ ]:
# Qarray[len(pax_list)][len(alt_routes)]

# totalPassengers = len(pax_list)
# totalRoutes = len(alt_routes)

Qarray = np.zeros((totalPassengers, totalRoutes), dtype=int)


def objectiveFunction(passenger, route):

    timeChange = route[3] - passenger[3]
    flightChange = (len(route)-2) - 1

    return (10 * flightChange + 3 * timeChange)
    
    
def cost(Qarray, pax_list, alt_routes):
    totalPassengers = len(pax_list)
    totalRoutes = len(alt_routes)
    duplicate = 0
    totalcost = -100
    for x in range TotalPassengers:
        if duplicate == 0:
            totalcost + 100
        duplicate = 0
        
        for y in range TotalConnections:
            if Qarray[x][y] == 1:
                if duplicate ==  1:
                    return 100000
                duplicate + 1
                passenger = pax_list[x]
                connection = alt_routes[y]
                totalcost += objectiveFunction(passenger, connection)

    return totalcost

